# P18b — CORAL / MMD Feature Alignment (main DA lever)

**One change vs P17:** add `λ·align(h_source, h_target_unlabeled)` (CORAL default, MMD
optional) on the learned features of BOTH models. No target labels; selection on
source-val; DANN-over-sources kept. Read `README_P18b.md` and `../PROTOCOL_DA_LODO.md`.

Two arms — **source-only** (= P17 lower bound) and **CORAL** — then compare with the
keep/revert gate. `--align_lambda` is the key knob (tune on source-val).

> **Final_PROJECT note (updated 2026-09-26).** All paths now come from
> `Final_PROJECT/project_paths.py` via the first code cell below — the only thing a new user may
> need to edit is the DAGHAR data path in that file (and `FINAL_PROJECT_ROOT` below, only if the
> folder isn't found automatically). **Saved cell outputs are from the original thesis run**, when
> this code lived at `/content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/P17_DA_LODO/Project_DA_P18b_FeatAlign`; they are kept as the record of that run.
> Note: `run_lodo.py` / the P18b runners are **not** resumable — re-running a training cell retrains and rewrites that run's output folder.
>
> ⚠️ **Historical record — do not re-run.** This notebook ran the whole P18b phase (GNN + ConvNet, source-only and CORAL arms). Final_PROJECT keeps only the ConvNet part that feeds the final model (`convnet/`, `da_baseline/convnet_predict.py`, and the source-only `base_s42/s1/s2` predictions), so the GNN/CORAL cells here would fail. To (re)produce the frozen ConvNet predictions use **`Run_Frozen_ConvNet_Colab.ipynb`** in this folder instead.

## 1. Mount Drive & set paths

In [ ]:
# ── Mount Drive & load Final_PROJECT paths (single source: Final_PROJECT/project_paths.py) ──
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob

# EDIT ONLY IF you uploaded Final_PROJECT somewhere else on your Drive
# (if this path doesn't exist, the folder is searched for automatically).
FINAL_PROJECT_ROOT = "/content/drive/MyDrive/Thesis/SO_Experiments/Final_PROJECT"

if not os.path.isfile(os.path.join(FINAL_PROJECT_ROOT, "project_paths.py")):
    hits = []
    for depth in range(0, 5):
        hits = glob.glob("/content/drive/MyDrive/" + "*/" * depth + "Final_PROJECT/project_paths.py")
        if hits: break
    assert hits, "Final_PROJECT not found on Drive — set FINAL_PROJECT_ROOT above"
    FINAL_PROJECT_ROOT = os.path.dirname(hits[0]); print("auto-detected Final_PROJECT at", FINAL_PROJECT_ROOT)
sys.path.insert(0, FINAL_PROJECT_ROOT)
import project_paths as PP            # the DAGHAR data path is set inside project_paths.py

PROJECT_PATH = PP.activate("convnet_b")   # exports DATA_PATH, checks data, cd's into this component
DATA_PATH    = PP.DATA_PATH

## 2. Setup — install PyG, verify data

In [2]:
import setup_colab
setup_colab.install_deps()
assert setup_colab.verify(), "fix DATA_PATH"

Installing PyTorch Geometric ...
Done.

Verifying imports ...
  OK torch 2.11.0+cu128  CUDA=True
     GPU Tesla T4
  OK torch_geometric 2.8.0

Checking data at: /content/drive/MyDrive/HAR-Datasets/DAGHAR/standardized_view
  OK KuHar              3 CSVs
  OK MotionSense        3 CSVs
  OK RealWorld-Thigh    3 CSVs
  OK RealWorld-Waist    3 CSVs
  OK UCI                3 CSVs
  OK WISDM              3 CSVs


## 3. Smoke test — WISDM, 2 epochs, both arms (+ alignment + gate check)
Confirms the CORAL path runs (you should see a non-zero `align` term in the logs) and
the compare script works end-to-end.

In [3]:
!python -m da_baseline.gnn_predict     --data_path "$DATA_PATH" --targets WISDM --epochs 2 --log_every 1 --out_dir results/probs_dg
!python -m da_baseline.convnet_predict --data_path "$DATA_PATH" --targets WISDM --epochs 2 --log_every 1 --out_dir results/probs_dg
!python -m da_baseline.gnn_predict     --data_path "$DATA_PATH" --targets WISDM --epochs 2 --log_every 1 --align coral --align_lambda 1.0 --out_dir results/probs_align
!python -m da_baseline.convnet_predict --data_path "$DATA_PATH" --targets WISDM --epochs 2 --log_every 1 --align coral --align_lambda 1.0 --out_dir results/probs_align
!python da_baseline/run_compare.py --out results/p18b_compare_smoke.json --da_label coral

[GNN] P18b | mode=source-only baseline (DG-norm) | P9 recipe | device-data=/content/drive/MyDrive/HAR-Datasets/DAGHAR/standardized_view | targets=['WISDM'] | out=results/probs_dg
[GNN] NOTE: source-only = P17 DG lower bound (target-unlabeled loaded, unused).
[LODO] target = WISDM
[LODO] sources = ['KuHar', 'MotionSense', 'RealWorld-Thigh', 'RealWorld-Waist', 'UCI']
[LODO] feature_mode = hybrid  normalize_mode = per_channel
[LODO] samples: train=28040  val=4900  test=2604
    feature_mode=hybrid, feat_dim=90, normalize_mode=per_channel
    building 28040 sensor graphs (6 nodes, 30 edges, 90 features each)...
    built 5000/28040 graphs
    built 10000/28040 graphs
    built 15000/28040 graphs
    built 20000/28040 graphs
    built 25000/28040 graphs
    done: 28040 graphs built
[LODO-DA] target-unlabeled (target train, labels withheld): 8748 graphs  [normalized with source-train stats]
  [gnn:WISDM] source-train=28040  source-val=4900  target-test=2604  target-unlabeled=8748 (NOT traine

## 4a. Baseline arm — source-only, all folds  →  results/probs_dg/
Reproduces the P17 lower bound. If you have trustworthy P17 baseline dumps already,
copy them into `results/probs_dg/` and skip this cell.

In [4]:
!python -m da_baseline.gnn_predict     --data_path "$DATA_PATH" --targets all --epochs 100 --log_every 5
!python -m da_baseline.convnet_predict --data_path "$DATA_PATH" --targets all --epochs 100 --log_every 5

[GNN] P18b | mode=source-only baseline (DG-norm) | P9 recipe | device-data=/content/drive/MyDrive/HAR-Datasets/DAGHAR/standardized_view | targets=['KuHar', 'MotionSense', 'RealWorld-Thigh', 'RealWorld-Waist', 'UCI', 'WISDM'] | out=/content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/P17_DA_LODO/Project_DA_P18b_FeatAlign/results/probs_dg
[GNN] NOTE: source-only = P17 DG lower bound (target-unlabeled loaded, unused).
[LODO] target = KuHar
[LODO] sources = ['MotionSense', 'RealWorld-Thigh', 'RealWorld-Waist', 'UCI', 'WISDM']
[LODO] feature_mode = hybrid  normalize_mode = per_channel
[LODO] samples: train=35396  val=5412  test=144
    feature_mode=hybrid, feat_dim=90, normalize_mode=per_channel
    building 35396 sensor graphs (6 nodes, 30 edges, 90 features each)...
    built 5000/35396 graphs
    built 10000/35396 graphs
    built 15000/35396 graphs
    built 20000/35396 graphs
    built 25000/35396 graphs
    built 30000/35396 graphs
    built 35000/35396 graphs
    done: 35396 graphs bu

## 4b. DA arm — CORAL alignment, all folds  →  results/probs_align/
`--align_lambda 1.0` is a starting point. If source-val accuracy drops a lot, lower it
(e.g. 0.3); if `align` barely moves, raise it (e.g. 3). Swap `coral`→`mmd` to try MMD.

In [5]:
!python -m da_baseline.gnn_predict     --data_path "$DATA_PATH" --targets all --epochs 100 --log_every 5 --align coral --align_lambda 1.0
!python -m da_baseline.convnet_predict --data_path "$DATA_PATH" --targets all --epochs 100 --log_every 5 --align coral --align_lambda 1.0

[GNN] P18b | mode=align=coral λ=1 | P9 recipe | device-data=/content/drive/MyDrive/HAR-Datasets/DAGHAR/standardized_view | targets=['KuHar', 'MotionSense', 'RealWorld-Thigh', 'RealWorld-Waist', 'UCI', 'WISDM'] | out=/content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/P17_DA_LODO/Project_DA_P18b_FeatAlign/results/probs_align
[GNN] NOTE: CORAL aligns source↔target-unlabeled on the graph embedding. No target labels; selection on source-val; DANN-over-sources kept.
[LODO] target = KuHar
[LODO] sources = ['MotionSense', 'RealWorld-Thigh', 'RealWorld-Waist', 'UCI', 'WISDM']
[LODO] feature_mode = hybrid  normalize_mode = per_channel
[LODO] samples: train=35396  val=5412  test=144
    feature_mode=hybrid, feat_dim=90, normalize_mode=per_channel
    building 35396 sensor graphs (6 nodes, 30 edges, 90 features each)...
    built 5000/35396 graphs
    built 10000/35396 graphs
    built 15000/35396 graphs
    built 20000/35396 graphs
    built 25000/35396 graphs
    built 30000/35396 graphs
    bu

## 5. Compare + gate

In [6]:
!python da_baseline/run_compare.py --da_label coral


  P18b — source-only  vs  coral (feature alignment)

  Fold             source-on     coral      Δpp
  ----------------------------------------------
  KuHar             63.19%  63.89%   +0.69
  MotionSense       84.84%  85.50%   +0.66
  RealWorld-Thigh   74.40%  74.84%   +0.45
  RealWorld-Waist   77.28%  75.85%   -1.43
  UCI               86.23%  87.54%   +1.30
  WISDM             72.54%  69.51%   -3.03
  ----------------------------------------------
  MEAN              76.41%  76.19%   -0.23

  Fusion weight (w on GNN): source-only w=0.6 | coral w=0.6

  GATE: keep if mean Δ >= +1.0pp AND no fold drops > 2pp
        mean Δ = -0.23pp | worst fold = WISDM (-3.03pp)
  VERDICT: REVERT or retune lambda; if no lambda helps, go to P18c (DANN target).

  Reference: P17 source-only fused mean = 76.88%

  saved /content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/P17_DA_LODO/Project_DA_P18b_FeatAlign/results/p18b_compare.json


---
### Reading the verdict
- **KEEP** (mean Δ ≥ +1.0pp, no fold < −2pp) → carry into **P18c** (DANN source→target) / tune λ.
- **REVERT** → retune λ; if nothing clears the gate, go to P18c.
Borderline? Seed-average 2–3 runs (KuHar N=144 is noisy) before deciding.
Watch the per-fold pattern: the goal is lifting KuHar/WISDM **without** dropping RW-Waist/UCI/MotionSense.